# PyRosetta 入门教程

从上到下跑一遍，走完 PyRosetta 的核心流程：

```
01 Pose             数据结构：一个分子体系的当前状态
02 ScoreFunction    打分：把结构变成一个数
03 Mover            操作：修改结构（FastRelax）
04 ResidueSelector  限定范围：只处理关心的那部分
05 InterfaceAnalyzer 实战：算抗体-抗原的结合能 ΔG
```

⚠️ 03 的两次 relax 合计约 16 分钟。只想快速过一遍的话可以跳过它们，**04 和 05 不依赖其结果**。

生产用法（批量评估、ΔΔG 突变分析）见 `Rosetta_Functions.md`。

## 01  Pose

Pose 是 Rosetta 表示「一个分子体系当前状态」的中心对象，里面装着：构象（xyz + 二面角）、序列与化学、链拓扑、上次打分的能量、以及 PDBInfo（原始 PDB 的链号与残基编号）。

两个要点：

1. **Pose 是可变的** —— 所有 Mover 都是原地修改传进去的 Pose，不返回新对象。
2. **两套残基编号** —— Rosetta 内部从 1 连续编到 N、无视链边界；PDB 文件按链分别计数。两者靠 PDBInfo 换算。

In [1]:
import pyrosetta

pyrosetta.init('-mute all')    # 每个进程只需 init 一次，加载数据库要十几秒

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python311.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
pose = pyrosetta.pose_from_sequence('AAAGGGKKK')    # 不用文件，直接从序列造一个 Pose

print(pose.total_residue())          # 残基总数
print(pose.sequence())               # 序列
print(pose.residue(3).name3())       # 第 3 个残基的三字母名（注意从 1 开始数）
print(pose.phi(3), pose.psi(3))      # 第 3 个残基的主链二面角

9
AAAGGGKKK
ALA
180.0 180.0


In [3]:
print(pose)    # Pose 的摘要：序列、折叠树、链信息

PDB file name: AAAGGGKK
Total residues: 9
Sequence: AAAGGGKKK
Fold tree:
FOLD_TREE  EDGE 1 9 -1 


### 01-2  加载真实复合物

`8hpu_M_N_A.pdb` 是一个抗体-抗原复合物：H 链（VH 121 aa）、L 链（VL 109 aa）、A 链（抗原 194 aa），共 424 残基。
只有 ATOM 记录，无水、无配体、无 altloc。

In [4]:
pdb = '/data/lmk/rosetta_inputs/8hpu_M_N_A.pdb'
pose = pyrosetta.pose_from_pdb(pdb)

print('总残基数:', pose.total_residue())
print('链数:', pose.num_chains())
print('前 30 个残基:', pose.sequence()[:30])

总残基数: 424
链数: 3
前 30 个残基: VQLVESGGGLVQPGGSLRLSCAASEITVSS


**两套编号的对照** —— PDBInfo 是连接 Rosetta 内部编号和原始 PDB 编号的桥梁。

In [5]:
info = pose.pdb_info()

for ch in range(1, pose.num_chains() + 1):
    b, e = pose.chain_begin(ch), pose.chain_end(ch)    # 该链在 Rosetta 编号里的起止
    print(f'链 {info.chain(b)}   Rosetta {b:>4} - {e:<4}   PDB {info.number(b):>4} - {info.number(e):<4}')

链 H   Rosetta    1 - 121    PDB    1 - 121 
链 L   Rosetta  122 - 230    PDB    1 - 109 
链 A   Rosetta  231 - 424    PDB    1 - 194 


可以看到每条链的 PDB 编号都从头开始，而 Rosetta 编号一路连续往下数。

下面是双向换算，**选残基必用**。

In [6]:
print('L 链第 30 位  ->  Rosetta 编号', info.pdb2pose('L', 30))    # PDB -> Rosetta
print('Rosetta 150   ->  PDB', info.pose2pdb(150))               # Rosetta -> PDB

i = info.pdb2pose('L', 30)
print('核对:', pose.residue(i).name3(), '在', info.pose2pdb(i))

L 链第 30 位  ->  Rosetta 编号 151
Rosetta 150   ->  PDB 29 L 
核对: SER 在 30 L 


In [7]:
chains = pose.split_by_chain()    # 按链拆成独立的 Pose

for k in range(1, len(chains) + 1):
    print(k, chains[k].total_residue(), chains[k].sequence()[:20])

1 121 VQLVESGGGLVQPGGSLRLS
2 109 DIQMTQSPSSLSASVGDRVS
3 194 NLCPFDEVFNATRFASVYAW


## 02  ScoreFunction

打分函数：输入一个 Pose，输出一个数（REU）。所有判断——构象好不好、突变有没有改善、binder 值不值得做——最终都归结为比较这个数。

总分 = Σ（权重 × 能量项）。REF2015 共十几项，半物理半统计：一部分来自物理公式（范德华、静电），一部分来自 PDB 数据库统计（rotamer 频率、Ramachandran 分布）。

| 类别 | 能量项 | 含义 |
| :--- | :--- | :--- |
| 范德华 | `fa_atr` / `fa_rep` | 原子间吸引 / 排斥（碰撞惩罚） |
| 溶剂化 | `fa_sol` `lk_ball_wtd` | 把极性基团埋进疏水环境的代价 |
| 静电 | `fa_elec` | 带电与极性基团间的库仑作用 |
| 氢键 | `hbond_sr_bb` `hbond_lr_bb` `hbond_bb_sc` `hbond_sc` | 按主链 / 侧链组合分四类 |
| 构象统计 | `fa_dun` | 侧链构象在 rotamer 库里常不常见 |
|  | `rama_prepro` `p_aa_pp` `omega` | 主链二面角是否落在允许区 |
| 参考态 | `ref` | 每种氨基酸的基线能量，做设计时防止过度偏好 |

两个性质：

1. **单位是 REU**，与 kcal/mol 数量级接近但未经标定 —— 不要写成 kcal/mol，也不要换算 Kd。
2. **是广延量**，蛋白越大总分越负，所以总分不能跨体系比较。评估 binder 要用分离前后的差值，而不是总分。

In [8]:
scorefxn = pyrosetta.get_fa_scorefxn()    # 默认就是 REF2015

total = scorefxn(pose)
print('总分:', total, 'REU')

总分: 313.25165813278215 REU


**各能量项的 REU 分解** —— 从 Python 侧读 `pose.energies()` 再乘以权重。

In [9]:
import pandas as pd

scorefxn(pose)                          # 必须先打分，energies 才有效
e = pose.energies().total_energies()    # 各能量项的未加权总和

rows = []
for st in scorefxn.get_nonzero_weighted_scoretypes():
    raw = e[st]
    w = scorefxn.get_weight(st)
    rows.append({'能量项': str(st).replace('ScoreType.', ''),
                 '原始值': round(raw, 2),
                 '权重': w,
                 '加权贡献': round(raw * w, 2)})

df = pd.DataFrame(rows).sort_values('加权贡献', ascending=False, ignore_index=True)

print('加权合计:', round(df['加权贡献'].sum(), 2),
      '  |  scorefxn(pose):', round(scorefxn(pose), 2))    # 自检：两者应当相等
df

加权合计: 313.25   |  scorefxn(pose): 313.25


,能量项,原始值,权重,加权贡献
0,fa_sol,1316.69,1.000,1316.69
1,fa_dun,1347.38,0.700,943.17
2,fa_rep,1215.08,0.550,668.30
3,ref,178.10,1.000,178.10
4,fa_intra_sol_xover4,84.25,1.000,84.25
5,pro_close,54.58,1.250,68.22
6,rama_prepro,143.09,0.450,64.39
7,omega,149.10,0.400,59.64
8,fa_intra_rep,972.70,0.005,4.86
9,yhh_planarity,0.00,0.625,0.00


## 03  Mover 与 FastRelax

**Mover** 是「任何修改 Pose 的操作」的统一抽象，接口只有一个：`mover.apply(pose)`。重排侧链、最小化、突变、对接、relax 全都是 Mover。

⚠️ **Mover 原地修改传进去的 Pose，不返回新对象。** 要保留原结构做对比，必须先 `pose.clone()`。

**FastRelax** 循环交替两件事：`repack` 重排侧链 rotamer（压低 `fa_dun`）、`minimize` 梯度最小化（压低 `fa_rep` / `omega` / `rama_prepro`）。过程中把 `fa_rep` 的权重从很低逐步升回 0.55 —— 先放松排斥力让纠缠的原子错开，再逐步收紧。

FastRelax 有两种常用配置，按目的二选一：

| | 主链 | 适用场景 |
| :--- | :--- | :--- |
| **03-1 无约束** | 完全自由，可漂移 1 Å 以上 | 只关心该结构的能量下限；或一批结构同等处理后互相排序 |
| **03-2 带坐标约束** | 拴在初始位置附近 | 要评估输入结构本身的界面，不希望构象被改动 |

抗体 / binder 设计的评估通常用后者。

**FastRelax** 是一种具体的 Mover。也就是说：

```
Mover
├─ FastRelax
├─ MinMover
├─ PackRotamersMover
├─ MutateResidue
└─ DockingProtocol
```

所有 Mover 都提供相同的接口：

```python
mover.apply(pose)
```

例如：

```python
relax = FastRelax()
relax.apply(pose)
```

### 03-1  无约束的 relax

不加任何限制，让 FastRelax 自由寻找能量最低的构象，得到的是该结构在 REF2015 下能达到的能量下限。代价是主链会漂移，relax 后的构象可能已经不是输入的那一个。

In [10]:
import time
from pyrosetta.rosetta.protocols.relax import FastRelax

pose_relaxed = pose.clone()    # Mover 原地改，必须先留底

fr = FastRelax()
fr.set_scorefxn(scorefxn)

t0 = time.time()
fr.apply(pose_relaxed)         # 424 残基单核约 5-20 分钟，耐心等
print(f'耗时 {time.time() - t0:.0f} 秒')

耗时 732 秒


In [11]:
from pyrosetta.rosetta.core.scoring import CA_rmsd

before, after = scorefxn(pose), scorefxn(pose_relaxed)

print(f'relax 前     {before:10.2f} REU')
print(f'relax 后     {after:10.2f} REU')
print(f'变化         {after - before:10.2f} REU')
print(f'主链 CA RMSD {CA_rmsd(pose, pose_relaxed):10.2f} Å')    # 结构被挪动了多少

relax 前         313.25 REU
relax 后       -1187.88 REU
变化           -1501.13 REU
主链 CA RMSD       1.33 Å


In [12]:
def energy_table(p):    # 某个 Pose 的各项加权贡献
    scorefxn(p)
    e = p.energies().total_energies()
    return {str(st).replace('ScoreType.', ''): round(e[st] * scorefxn.get_weight(st), 2)
            for st in scorefxn.get_nonzero_weighted_scoretypes()}

tbl = pd.DataFrame({'relax 前': energy_table(pose),
                    'relax 后': energy_table(pose_relaxed)})
tbl['变化'] = (tbl['relax 后'] - tbl['relax 前']).round(2)
tbl.sort_values('变化')    # 降幅最大的排最前

,relax 前,relax 后,变化
fa_dun,943.17,467.32,-475.85
fa_rep,668.30,315.87,-352.43
fa_elec,-445.41,-661.55,-216.14
fa_atr,-2293.17,-2381.00,-87.83
pro_close,68.22,1.37,-66.85
hbond_bb_sc,-32.26,-98.59,-66.33
hbond_sc,-28.68,-74.78,-46.10
p_aa_pp,-61.24,-100.09,-38.85
rama_prepro,64.39,27.81,-36.58
hbond_sr_bb,-29.61,-58.62,-29.01


### 03-2  固定主链的 relax

用 **MoveMap** 声明哪些自由度可以动：主链完全锁死，只放开侧链。评估 AF3 / RFdiffusion 给出的设计时通常用这个 —— 要判断的就是模型给出的那个主链构象，不该让 Rosetta 改它，只需要把侧链摆顺、消掉碰撞。

`MoveMap()` 默认所有自由度都是关的，`set_chi(True)` 只放开侧链，主链和 jump 保持锁定，所以 **CA RMSD 必然为 0**。主链自由度全锁后最小化的维度大幅减少，比 03-1 快不少。

固定主链只能回收约一半能量，因为主链层面的应变修不了：`hbond_sr_bb`、`hbond_lr_bb`、`omega`、`rama_prepro`、`p_aa_pp`、`ref` 六个纯主链项**数值与输入完全相同** —— 主链冻结后它们在数学上不可能变化。

In [13]:
from pyrosetta.rosetta.core.kinematics import MoveMap

mm = MoveMap()
mm.set_bb(False)      # 主链固定 
mm.set_chi(True)      # 只放开侧链

pose_fixbb = pose.clone()

fr3 = FastRelax()
fr3.set_scorefxn(scorefxn)    # 不需要约束项，用原始打分函数即可
fr3.set_movemap(mm)

t0 = time.time()
fr3.apply(pose_fixbb)
print(f'耗时 {time.time() - t0:.0f} 秒')

耗时 123 秒


In [14]:
rows = []
for name, p in [('原始', pose), ('03-1 无约束', pose_relaxed), ('03-2 固定主链', pose_fixbb)]:
    rows.append({'结构': name,
                 '总分 (REU)': round(scorefxn(p), 2),
                 'CA RMSD (Å)': round(CA_rmsd(pose, p), 2)})

pd.DataFrame(rows)

,结构,总分 (REU),CA RMSD (Å)
0,原始,313.25,0.00
1,03-1 无约束,-1187.88,1.33
2,03-2 固定主链,-475.10,0.00


In [15]:
pd.DataFrame({'原始': energy_table(pose),
              '无约束': energy_table(pose_relaxed),
              '固定主链': energy_table(pose_fixbb)})

,原始,无约束,固定主链
fa_atr,-2293.17,-2381.00,-2244.69
fa_rep,668.30,315.87,483.56
fa_sol,1316.69,1292.09,1241.68
fa_intra_rep,4.86,4.45,4.57
fa_intra_sol_xover4,84.25,68.83,67.28
lk_ball_wtd,-39.61,-53.34,-55.21
fa_elec,-445.41,-661.55,-519.28
pro_close,68.22,1.37,4.20
hbond_sr_bb,-29.61,-58.62,-29.61
hbond_lr_bb,-143.04,-164.06,-143.04


## 04  ResidueSelector

`selector.apply(pose)` 返回一个 1 起始的布尔向量（长度 = 残基总数），标出哪些残基入选。

它本身不修改任何东西，作用是**告诉 Mover 只处理哪些位置** —— 只 relax 界面、只突变 CDR、只重排某条链的侧链。

核心性质是**可组合**：用 `And` / `Or` / `Not` 把简单选择器拼成复杂条件，不必自己写循环算距离。

| 选择器 | 选出 |
| :--- | :--- |
| `ChainSelector('A')` | 指定链 |
| `ResidueIndexSelector('1-10,15')` | 指定编号（Rosetta 编号） |
| `NeighborhoodResidueSelector(sel, 8.0, False)` | 距某组残基 8 Å 以内 |
| `InterGroupInterfaceByVectorSelector` | 两组之间的界面（更严格，考虑侧链朝向） |
下面用 8 Å 邻域选出的 paratope 有 26 个残基，**全部落在六个 CDR 上**（仅 H 链第 1 位那个框架残基例外）—— 纯几何条件独立地把 CDR 找了出来，也反过来说明这个复合物的结合模式正常。

In [16]:
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector
from pyrosetta.rosetta.core.select import get_residues_from_subset

sel_A = ChainSelector('A')
mask = sel_A.apply(pose)                      # 1 起始的布尔向量，长度 = 残基总数
idx = list(get_residues_from_subset(mask))    # 转成 Rosetta 编号的列表

print('抗原链残基数:', len(idx))
print('前 5 个 Rosetta 编号:', idx[:5])
print('对应 PDB 编号:', [info.pose2pdb(i).strip() for i in idx[:5]])

抗原链残基数: 194
前 5 个 Rosetta 编号: [231, 232, 233, 234, 235]
对应 PDB 编号: ['1 A', '2 A', '3 A', '4 A', '5 A']


In [17]:
from pyrosetta.rosetta.core.select.residue_selector import NeighborhoodResidueSelector

near = NeighborhoodResidueSelector(ChainSelector('A'), 8.0, False)    # 距抗原 8 Å 内，不含抗原自身
idx_near = list(get_residues_from_subset(near.apply(pose)))

print('抗原 8 Å 邻域内的残基数:', len(idx_near))
print('按链分布:', {ch: sum(1 for i in idx_near if info.chain(i) == ch) for ch in 'HLA'})

抗原 8 Å 邻域内的残基数: 26
按链分布: {'H': 19, 'L': 7, 'A': 0}


In [18]:
from pyrosetta.rosetta.core.select.residue_selector import AndResidueSelector, OrResidueSelector

hl = OrResidueSelector(ChainSelector('H'), ChainSelector('L'))    # 抗体两条链
paratope = AndResidueSelector(near, hl)                          # 靠近抗原 且 属于抗体 = 结合面

idx_p = list(get_residues_from_subset(paratope.apply(pose)))
print('抗体侧界面残基数:', len(idx_p))
print()

for i in idx_p:
    print(f'  {info.pose2pdb(i).strip():<8} {pose.residue(i).name3()}')

抗体侧界面残基数: 26

  1 H      VAL
  25 H     GLU
  26 H     ILE
  27 H     THR
  29 H     SER
  30 H     SER
  31 H     ASN
  51 H     TYR
  52 H     PRO
  53 H     GLY
  54 H     GLY
  55 H     SER
  57 H     PHE
  98 H     SER
  99 H     GLY
  100 H    GLY
  101 H    PHE
  103 H    LEU
  105 H    GLU
  27 L     GLN
  28 L     SER
  30 L     SER
  56 L     SER
  92 L     TYR
  94 L     THR
  95 L     PRO


**选出来之后怎么用** —— ResidueSelector 只负责「描述哪些残基」，本身不会去限制 relax。要真正用上，得把选择结果写进 MoveMap：

```
ResidueSelector  →  MoveMap  →  FastRelax
```

⚠️ 必须放开**界面两侧** —— 抗体的 paratope 加抗原的 epitope。只放开一侧的话，另一侧的侧链动不了，两边挤在一起的碰撞依然消不掉。

局部 relax 之后总分仍然很难看（蛋白其余部分没被处理），但**不影响界面评估** —— 结合能是「复合物 − 分开的两部分」的差值，未处理的应变在两边都出现，相减时抵消。

跑完之后有两条规律值得记住：

1. **界面承担了不成比例的应变** —— 界面那 53 个残基只占全蛋白 12.5%，却回收了全侧链 relax 44% 的能量。界面堆积最紧密，补氢也最容易出问题。
2. **自由度减少不等比例地省时间** —— 残基数降到 1/8，耗时只降到 1/2。FastRelax 每轮仍要给整个 Pose 打分、构建 rotamer 集合，固定开销占比很大。**估算批量任务耗时时要考虑这一点。**

In [19]:
epitope = AndResidueSelector(
    NeighborhoodResidueSelector(hl, 8.0, False),    # 靠近抗体
    ChainSelector('A'))                             # 且属于抗原

idx_e = list(get_residues_from_subset(epitope.apply(pose)))
idx_iface = sorted(set(idx_p) | set(idx_e))         # 界面两侧合并

print('paratope 抗体侧:', len(idx_p))
print('epitope  抗原侧:', len(idx_e))
print('界面合计:', len(idx_iface), '/', pose.total_residue())

paratope 抗体侧: 26
epitope  抗原侧: 27
界面合计: 53 / 424


In [20]:
import time
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.relax import FastRelax

mm_local = MoveMap()
mm_local.set_bb(False)          # 主链全部固定
mm_local.set_chi(False)         # 侧链默认也固定
for i in idx_iface:
    mm_local.set_chi(i, True)   # 只放开界面残基的侧链

pose_local = pose.clone()

fr4 = FastRelax()
fr4.set_scorefxn(scorefxn)
fr4.set_movemap(mm_local)

t0 = time.time()
fr4.apply(pose_local)
print(f'耗时 {time.time() - t0:.0f} 秒')

耗时 56 秒


In [21]:
from pyrosetta.rosetta.core.scoring import CA_rmsd

cands = [('原始', pose)]
if 'pose_fixbb' in globals():                       # 03-2 跑过才有
    cands.append(('03-2 全侧链', pose_fixbb))
cands.append(('04-2 仅界面', pose_local))

rows = [{'结构': name,
         '总分 (REU)': round(scorefxn(p), 2),
         'CA RMSD (Å)': round(CA_rmsd(pose, p), 2)} for name, p in cands]

pd.DataFrame(rows)

,结构,总分 (REU),CA RMSD (Å)
0,原始,313.25,0.0
1,03-2 全侧链,-475.10,0.0
2,04-2 仅界面,-33.86,0.0


## 05  InterfaceAnalyzer —— 算结合能 ΔG

前四节的东西在这里汇合：用 04 局部 relax 过的 `pose_local` 当输入，算抗体与抗原之间的结合能。

```
ΔG_binding = E(复合物) − E(把两部分分开后)
```

`set_pack_separated(True)` 让两部分分开后重新 repack 侧链 —— 模拟解离时侧链舒展的真实过程，对结果影响很大。

⚠️ 它是 Mover，`apply()` 会修改传进去的 Pose（内部要把链分开再合回来），照例先 `clone()`。

In [22]:
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

pose_ia = pose_local.clone()            # 04 局部 relax 后的结构

ia = InterfaceAnalyzerMover(2)          # jump 2 连着抗原 A 链，沿它分开 = HL vs A
ia.set_scorefunction(scorefxn)
ia.set_pack_separated(True)             # 分开后重新 repack

t0 = time.time()
ia.apply(pose_ia)
print(f'耗时 {time.time() - t0:.0f} 秒')
print()
print('dG_separated =', round(ia.get_interface_dG(), 2), 'REU')

耗时 3 秒

dG_separated = -76.91 REU


## 小结

跑完这份 tutorial，你应该能：

| 章节 | 会做什么 |
| :--- | :--- |
| **01 Pose** | 加载结构、在 Rosetta 与 PDB 两套编号之间换算、按链拆分 |
| **02 ScoreFunction** | 给结构打分，并把总分拆解到各能量项 |
| **03 Mover** | 用 FastRelax 处理结构，按目的决定放不放开主链 |
| **04 ResidueSelector** | 圈定界面残基，用 MoveMap 把 relax 限制在指定范围 |
| **05 InterfaceAnalyzer** | 算出抗体与抗原之间的结合能 ΔG |

这五块拼起来就是 PyRosetta 的完整工作模式：**数据（Pose）→ 打分（ScoreFunction）→ 操作（Mover）→ 限定范围（ResidueSelector）→ 得到结论**。

这五节的概念说明、判断依据和实测数据都在本 notebook 的 markdown 与输出里。

**下一步**：批量评估、ΔΔG 突变分析等生产用法，见 `Rosetta_Functions.md`。